<div style="width: 98%; max-width: 1200px; margin: 0 auto; font-family: Helvetica, Arial, sans-serif;">

<div style="
        background-color: #141E30;
        background-image: linear-gradient(to right, #141E30, #243B55);
        border-radius: 10px;
        padding: 25px;
        text-align: center;
        margin-bottom: 20px;
        border: 1px solid #2c3e50;
        box-shadow: 0 4px 8px rgba(0,0,0,0.3);
    ">
        <h1 style="color: white; margin: 0; text-transform: uppercase; font-size: 24px; font-weight: normal; letter-spacing: 2px;">
            TP 4 : Intégration numérique & Électrostatique
        </h1>
        <h3 style="color: #a8d0e6; margin-top: 10px; margin-bottom: 0; font-weight: normal; font-style: italic; font-size: 16px;">
            Dynamique épidémiologique SIS et condensateur électronique
        </h3>
    </div>

<div style="background-color: #141E30; border: 2px solid #a8d0e6; border-radius: 10px; padding: 20px; color: white;">

### Équipe de recherche
**Alex Baker** - 537 050 929  
**Justine Jean** - 537 287 332  
**Nerimantas Caillat** - 537 396 153 

### Informations académiques
**Cours :** PHY-3500 – Physique numérique (H26)  
**Remise :** 7 avril 2026  
**Encadrement :** Pr. Antoine Allard, Thomas Labbé

</div>

</div>

### Objectifs et Contexte

<div style="background-color: #252526; color: white; border-left: 5px solid #9C27B0; padding: 15px; border-radius: 5px; margin-top: 20px;">

<strong style="color: #E1BEE7; font-size: 1.1em;">Contexte :</strong>
<br><br>
Ce travail pratique couvre deux sujets distincts de la physique numérique. La première partie porte sur l'intégration numérique d'équations différentielles ordinaires à travers le modèle épidémiologique SIS. La seconde partie aborde la résolution d'équations aux dérivées partielles elliptiques (équation de Laplace) à l'aide de méthodes itératives pour calculer le potentiel électrostatique d'un condensateur simplifié.

<strong style="color: #E1BEE7; font-size: 1.1em;">Objectifs :</strong>
<br>
1. **Implémenter et comparer** les intégrateurs d'Euler, Runge-Kutta 2 et Runge-Kutta 4 sur la dynamique SIS.
2. **Quantifier numériquement** l'ordre de convergence de chaque méthode d'intégration.
3. **Résoudre l'équation de Laplace** par les méthodes de Gauss-Seidel et Jacobi pour un condensateur 2D.
4. **Optimiser** le paramètre de relaxation $\omega$ et comparer les performances des différentes méthodes.

</div>

In [1]:
# Calcul scientifique
import numpy as np
from scipy.optimize import minimize_scalar
from scipy.stats import linregress

# Visualisation
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# Mesure de performance et système
import time
import platform

# Configuration esthétique des graphiques
plt.rcParams.update({
    'figure.figsize': (12, 7),
    'figure.dpi': 100,
    'font.size': 11,
    'font.family': 'serif',
    'axes.labelsize': 12,
    'axes.titlesize': 14,
    'axes.titleweight': 'bold',
    'axes.grid': True,
    'grid.alpha': 0.3,
    'lines.linewidth': 2,
    'legend.fontsize': 11,
    'legend.framealpha': 0.9,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
})

print("=" * 40)
print("   CONFIGURATION DU NOTEBOOK RÉUSSIE")
print("=" * 40)
print(f"   NumPy version    : {np.__version__}")
print(f"   Python version   : {platform.python_version()}")
print(f"   Plateforme       : {platform.system()} {platform.release()}")
print("=" * 40)

   CONFIGURATION DU NOTEBOOK RÉUSSIE
   NumPy version    : 2.4.1
   Python version   : 3.11.9
   Plateforme       : Windows 10


<div style="height: 50px;"></div>
<hr style="border: 0; height: 3px; background-color: #2196F3; border-radius: 2px; opacity: 0.7;">
<br />

# TP4.1 — Intégration numérique de la dynamique épidémiologique SIS

<div style="background-color: #1a2a3a; color: #a8d0e6; border-left: 5px solid #2196F3; padding: 15px; border-radius: 5px; font-size: 0.95em;">

La dynamique SIS (*Susceptible–Infected–Susceptible*) modélise la propagation d'un agent infectieux sans immunité permanente. Après adimensionnement, l'équation gouvernante est :

$$\frac{di}{d\tau} = (R_0 - 1)\,i - R_0\,i^2$$

où $i(\tau)$ est la fraction infectée, $\tau = \alpha t$ le temps adimensionnel, et $R_0 = \beta/\alpha$ le nombre de reproduction de base.

</div>

<div style="height: 30px;"></div>
<hr style="border: 0; height: 2px; background-color: #4CAF50; border-radius: 2px; opacity: 0.6;">

## Question a — États stationnaires

<div style="background-color: #252526; color: white; border-left: 5px solid #4CAF50; padding: 15px; border-radius: 5px;">

**Énoncé :** *Identifiez les deux états stationnaires $0 \leq i^*_{1,2} \leq 1$ pour lesquels $\dfrac{di}{d\tau} = 0$, de même que les conditions sur $R_0$ pour lesquelles ces états stationnaires sont possibles.*

</div>

### Dérivation analytique

On cherche les racines de :
$$\frac{di}{d\tau} = (R_0 - 1)\,i - R_0\,i^2 = i\left[(R_0 - 1) - R_0 i\right] = 0$$

| État stationnaire | Expression | Condition d'existence |
|:-----------------|:-----------|:----------------------|
| $i^*_1$ | $0$ | Toujours valide |
| $i^*_2$ | $\dfrac{R_0 - 1}{R_0} = 1 - \dfrac{1}{R_0}$ | $R_0 \geq 1$ (et $i^*_2 \leq 1$) |

**Interprétation physique :**
- Si $R_0 < 1$ : seul $i^*_1 = 0$ est un état stationnaire physiquement admissible. L'épidémie s'éteint.
- Si $R_0 > 1$ : $i^*_2 = 1 - 1/R_0$ est un état endémique stable. L'épidémie persiste.
- Si $R_0 = 1$ : les deux états coïncident en $i^* = 0$ (bifurcation transcritique).

<div style="height: 30px;"></div>
<hr style="border: 0; height: 2px; background-color: #4CAF50; border-radius: 2px; opacity: 0.6;">

## Question b — Solution analytique

<div style="background-color: #252526; color: white; border-left: 5px solid #4CAF50; padding: 15px; border-radius: 5px;">

**Énoncé :** *Obtenez une solution analytique $i(\tau)$ valide pour $i \geq 0$. On note $i(0) = i_0$.*

</div>

### Résolution analytique par séparation des variables

L'équation $\dfrac{di}{d\tau} = (R_0 - 1)i - R_0 i^2$ est une **équation de Bernoulli** (ou équation logistique). On sépare les variables et on intègre par décomposition en fractions partielles.

*Votre développement ici...*

La solution analytique est :

$$\boxed{i(\tau) = \frac{(R_0-1)\,i_0}{R_0\,i_0 + (R_0 - 1 - R_0\,i_0)\,e^{-(R_0-1)\tau}}}$$

> ⚠️ **Cas particuliers :** Pour $R_0 = 1$, l'expression ci-dessus est indéfinie — dériver la solution limitante. Pour $i_0 = 0$, la solution triviale $i(\tau) = 0$ est valide.

In [ ]:
def solution_analytique(tau, i0, R0):
    """
    Solution analytique de la dynamique SIS adimensionnée.
    
    Paramètres
    ----------
    tau : array-like
        Temps adimensionnel
    i0 : float
        Fraction infectée initiale
    R0 : float
        Nombre de reproduction de base
    
    Retourne
    --------
    i : ndarray
        Fraction infectée au temps tau
    """
    tau = np.asarray(tau)
    # TODO: Implémenter la solution analytique
    pass

# Test rapide
tau_test = np.linspace(0, 10, 200)
i_analytique = solution_analytique(tau_test, i0=0.01, R0=2.5)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(tau_test, i_analytique, color='#2196F3', label=r'$i(\tau)$ analytique')
ax.set_xlabel(r'$\tau$ (temps adimensionnel)', fontsize=12)
ax.set_ylabel(r'Fraction infectée $i$', fontsize=12)
ax.set_title(r'Solution analytique SIS ($i_0=0.01$, $R_0=2.5$)')
ax.legend()
plt.tight_layout()
plt.show()

<div style="height: 30px;"></div>
<hr style="border: 0; height: 2px; background-color: #4CAF50; border-radius: 2px; opacity: 0.6;">

## Question c — Implémentation des intégrateurs

<div style="background-color: #252526; color: white; border-left: 5px solid #4CAF50; padding: 15px; border-radius: 5px;">

**Énoncé :** *Implémentez les intégrateurs d'Euler, de Runge-Kutta d'ordre 2 et de Runge-Kutta d'ordre 4. Vos fonctions devront retourner les trajectoires $\{i_s\}_{s=0,\ldots,T}$ et les temps $\{\tau_s\}_{s=0,\ldots,T}$.*

</div>

In [ ]:
def f_SIS(i, R0):
    """
    Membre de droite de l'équation SIS adimensionnée : di/dtau = f(i, R0).
    """
    return (R0 - 1) * i - R0 * i**2

In [ ]:
def integrateur_euler(f, i0, R0, h, T):
    """
    Intégrateur d'Euler explicite.
    
    Paramètres
    ----------
    f   : callable, membre de droite f(i, R0)
    i0  : float, condition initiale
    R0  : float, nombre de reproduction de base
    h   : float, pas d'intégration
    T   : int, nombre total de pas
    
    Retourne
    --------
    taus : ndarray, shape (T+1,)
    is_  : ndarray, shape (T+1,)
    """
    taus = np.zeros(T + 1)
    is_ = np.zeros(T + 1)
    is_[0] = i0
    
    for s in range(T):
        # TODO: Implémenter le schéma d'Euler
        pass
    
    return taus, is_

In [ ]:
def integrateur_rk2(f, i0, R0, h, T):
    """
    Intégrateur de Runge-Kutta d'ordre 2 (méthode de Heun / RK2 classique).
    
    Paramètres
    ----------
    f   : callable
    i0  : float
    R0  : float
    h   : float
    T   : int
    
    Retourne
    --------
    taus : ndarray
    is_  : ndarray
    """
    taus = np.zeros(T + 1)
    is_ = np.zeros(T + 1)
    is_[0] = i0
    
    for s in range(T):
        # TODO: Implémenter RK2
        pass
    
    return taus, is_

In [ ]:
def integrateur_rk4(f, i0, R0, h, T):
    """
    Intégrateur de Runge-Kutta d'ordre 4.
    
    Paramètres
    ----------
    f   : callable
    i0  : float
    R0  : float
    h   : float
    T   : int
    
    Retourne
    --------
    taus : ndarray
    is_  : ndarray
    """
    taus = np.zeros(T + 1)
    is_ = np.zeros(T + 1)
    is_[0] = i0
    
    for s in range(T):
        # TODO: Implémenter RK4
        pass
    
    return taus, is_

In [ ]:
# Validation visuelle des trois intégrateurs
i0, R0 = 0.01, 2.5
h_test = 0.05
T_test = 300

tau_euler, i_euler = integrateur_euler(f_SIS, i0, R0, h_test, T_test)
tau_rk2, i_rk2 = integrateur_rk2(f_SIS, i0, R0, h_test, T_test)
tau_rk4, i_rk4 = integrateur_rk4(f_SIS, i0, R0, h_test, T_test)
i_exact = solution_analytique(tau_rk4, i0, R0)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Trajectoires
axes[0].plot(tau_rk4, i_exact, 'k-', lw=2.5, label='Analytique', zorder=5)
axes[0].plot(tau_euler, i_euler, '--', color='#e74c3c', label='Euler')
axes[0].plot(tau_rk2, i_rk2, '--', color='#f39c12', label='RK2')
axes[0].plot(tau_rk4, i_rk4, '--', color='#2ecc71', label='RK4')
axes[0].set_xlabel(r'$\tau$')
axes[0].set_ylabel(r'Fraction infectée $i$')
axes[0].set_title('Trajectoires numériques vs analytique')
axes[0].legend()

# Erreurs absolues
axes[1].semilogy(tau_euler, np.abs(i_euler - solution_analytique(tau_euler, i0, R0)), color='#e74c3c', label='Euler')
axes[1].semilogy(tau_rk2, np.abs(i_rk2 - solution_analytique(tau_rk2, i0, R0)), color='#f39c12', label='RK2')
axes[1].semilogy(tau_rk4, np.abs(i_rk4 - i_exact), color='#2ecc71', label='RK4')
axes[1].set_xlabel(r'$\tau$')
axes[1].set_ylabel(r'$|i_s - i(\tau_s)|$')
axes[1].set_title('Erreur absolue en fonction du temps')
axes[1].legend()

plt.tight_layout()
plt.show()

<div style="height: 30px;"></div>
<hr style="border: 0; height: 2px; background-color: #4CAF50; border-radius: 2px; opacity: 0.6;">

## Question d — Méthode du ratio doré : trouver $h$ pour une erreur cible $\delta$

<div style="background-color: #252526; color: white; border-left: 5px solid #4CAF50; padding: 15px; border-radius: 5px;">

**Énoncé :** *Implémentez la méthode du ratio doré pour trouver le pas $h$ tel que l'erreur $\varepsilon(h)$ soit bornée par $(1\pm 0.01)\delta$, avec*

$$\varepsilon(h) = \sqrt{\frac{1}{T+1}\sum_{s=0}^{T}\left(i_s - i(\tau_s)\right)^2}$$

*Identifiez $h$ pour plusieurs valeurs de $\delta \in [10^{-9}, 10^{-6}]$, pour plusieurs couples $(i_0, R_0)$ et pour chacun des trois intégrateurs.*

</div>

In [ ]:
def erreur_trajectoire(integrateur, f, i0, R0, h, T):
    """
    Calcule l'erreur quadratique moyenne entre la trajectoire numérique et analytique.
    
    Retourne
    --------
    epsilon : float
    """
    taus, is_ = integrateur(f, i0, R0, h, T)
    i_exact = solution_analytique(taus, i0, R0)
    return np.sqrt(np.mean((is_ - i_exact)**2))

In [ ]:
def methode_ratio_dore(integrateur, f, i0, R0, delta, T, tolerance=0.01,
                       h_min=1e-6, h_max=1.0):
    """
    Méthode du ratio doré (golden-section search) pour trouver h tel que
    epsilon(h) in [(1 - tolerance)*delta, (1 + tolerance)*delta].
    
    L'idée est de minimiser |epsilon(h) - delta| sur [h_min, h_max].
    
    Paramètres
    ----------
    integrateur : callable
    f           : callable
    i0          : float
    R0          : float
    delta       : float, erreur cible
    T           : int, nombre de pas
    tolerance   : float, tolérance relative (défaut 1%)
    h_min, h_max: float, intervalle de recherche
    
    Retourne
    --------
    h_opt : float
    """
    phi = (1 + np.sqrt(5)) / 2  # Nombre d'or
    resphi = 2 - phi
    
    # TODO: Implémenter la recherche par ratio doré
    pass

In [ ]:
# Paramètres de l'étude
T_sim = 500
deltas = np.logspace(-9, -6, 7)
couples = [(0.01, 2.5), (0.05, 1.5), (0.1, 3.0)]
integrateurs = {
    'Euler': integrateur_euler,
    'RK2'  : integrateur_rk2,
    'RK4'  : integrateur_rk4,
}

resultats = {}  # {(nom_integ, i0, R0): {delta: h_opt}}

for nom, integ in integrateurs.items():
    for (i0, R0) in couples:
        cle = (nom, i0, R0)
        resultats[cle] = {}
        for delta in deltas:
            h_opt = methode_ratio_dore(integ, f_SIS, i0, R0, delta, T_sim)
            resultats[cle][delta] = h_opt
            print(f"[{nom:5s}] i0={i0}, R0={R0} | delta={delta:.1e} → h={h_opt:.4e}")

<div style="height: 30px;"></div>
<hr style="border: 0; height: 2px; background-color: #4CAF50; border-radius: 2px; opacity: 0.6;">

## Question e — Ordre de convergence $d$ par régression linéaire

<div style="background-color: #252526; color: white; border-left: 5px solid #4CAF50; padding: 15px; border-radius: 5px;">

**Énoncé :** *À l'aide des résultats de d, obtenez numériquement l'ordre $d$ de l'erreur globale $\varepsilon(h) \propto h^d$ de chaque intégrateur via une régression linéaire sur $\log\varepsilon(h) \propto d\log h$.*

</div>

In [ ]:
from scipy.stats import linregress

fig, ax = plt.subplots(figsize=(10, 6))
couleurs = {'Euler': '#e74c3c', 'RK2': '#f39c12', 'RK4': '#2ecc71'}
ordres_theoriques = {'Euler': 1, 'RK2': 2, 'RK4': 4}

# Utilisation du couple de référence (i0=0.01, R0=2.5)
i0_ref, R0_ref = 0.01, 2.5

for nom, integ in integrateurs.items():
    cle = (nom, i0_ref, R0_ref)
    hs = np.array(list(resultats[cle].values()))
    ds = np.array(list(resultats[cle].keys()))
    
    # Régression log-log : log(delta) ~ d * log(h) + cst
    pente, intercept, r, _, _ = linregress(np.log(hs), np.log(ds))
    
    ax.loglog(hs, ds, 'o', color=couleurs[nom], ms=7, label=f'{nom} (données)')
    h_fit = np.logspace(np.log10(hs.min()), np.log10(hs.max()), 100)
    ax.loglog(h_fit, np.exp(intercept) * h_fit**pente, '--', color=couleurs[nom],
              label=f'{nom}: $d = {pente:.2f}$ (théorie: {ordres_theoriques[nom]})')

ax.set_xlabel(r'Pas $h$', fontsize=12)
ax.set_ylabel(r'Erreur $\varepsilon(h)$', fontsize=12)
ax.set_title(r"Ordre de convergence : $\log\varepsilon$ vs $\log h$")
ax.legend()
plt.tight_layout()
plt.show()

### Discussion des résultats

<div style="background-color: #1a2a3a; color: #e0e0e0; border-left: 5px solid #2196F3; padding: 15px; border-radius: 5px;">

| Méthode | Ordre théorique | Ordre mesuré | Commentaire |
|:--------|:---------------:|:------------:|:------------|
| Euler   | 1               | —            | *À compléter* |
| RK2     | 2               | —            | *À compléter* |
| RK4     | 4               | —            | *À compléter* |

*Votre analyse ici...*

</div>

<div style="height: 50px;"></div>
<hr style="border: 0; height: 3px; background-color: #FF9800; border-radius: 2px; opacity: 0.7;">
<br />

# TP4.2 — Un condensateur électronique

<div style="background-color: #2a1f10; color: #ffe0b2; border-left: 5px solid #FF9800; padding: 15px; border-radius: 5px; font-size: 0.95em;">

On considère un condensateur simplifié 2D constitué de deux plaques métalliques dans un boîtier conducteur. On cherche le potentiel électrostatique $V(x, y)$ satisfaisant l'**équation de Laplace** $\nabla^2 V = 0$ avec les conditions aux limites suivantes :

- **Parois de la boîte :** $V = 0$ V (mise à la terre)
- **Plaque gauche** (de hauteur 10 cm, centrée horizontalement à 2 cm du bord gauche) : $V = +1$ V
- **Plaque droite** (de hauteur 6 cm, centrée verticalement à 2 cm du bord droit) : $V = -1$ V

La grille de calcul est de **100 × 100 points**, avec une précision de convergence de $10^{-6}$ V.

</div>

In [ ]:
# ── Paramètres géométriques ───────────────────────────────────────────────
N = 100                  # Taille de la grille (N x N)
Lx, Ly = 10.0, 10.0      # Dimensions de la boîte (cm)
dx = Lx / (N - 1)        # Pas spatial en x
dy = Ly / (N - 1)        # Pas spatial en y
PRECISION = 1e-6         # Critère de convergence (V)

# Coordonnées physiques
x = np.linspace(0, Lx, N)
y = np.linspace(0, Ly, N)

def construire_masque_plaques(N, Lx, Ly):
    """
    Retourne deux tableaux booléens indiquant les positions des plaques +1V et -1V
    sur la grille N x N.
    
    Géométrie (cf. Figure 4.1) :
      - Boîte : 10 cm (hauteur) x 10 cm (largeur)
      - Plaque +1V : 10 cm de haut, à x = 2 cm du bord gauche
      - Plaque -1V :  6 cm de haut (centrée), à x = 8 cm (bord droit - 2 cm)
    """
    masque_pos = np.zeros((N, N), dtype=bool)
    masque_neg = np.zeros((N, N), dtype=bool)
    
    # TODO: Définir les indices des plaques d'après la figure 4.1
    # Plaque +1V : colonne à x = 2 cm, toute la hauteur
    # Plaque -1V : colonne à x = 8 cm, 6 cm centré
    
    return masque_pos, masque_neg

masque_pos, masque_neg = construire_masque_plaques(N, Lx, Ly)
print(f"Grille : {N}x{N}, dx={dx:.3f} cm, dy={dy:.3f} cm")
print(f"Nombre de points plaque +1V : {masque_pos.sum()}")
print(f"Nombre de points plaque -1V : {masque_neg.sum()}")

<div style="height: 30px;"></div>
<hr style="border: 0; height: 2px; background-color: #FF5722; border-radius: 2px; opacity: 0.6;">

## Question a — Méthode de Gauss-Seidel avec relaxation

<div style="background-color: #252526; color: white; border-left: 5px solid #FF5722; padding: 15px; border-radius: 5px;">

**Énoncé :** *Calculez le potentiel électrostatique par la méthode de Gauss-Seidel avec relaxation sur une grille 100×100. Précision requise : $10^{-6}$ V. Affichez le résultat sous forme de carte thermique.*

</div>

In [ ]:
def gauss_seidel(N, masque_pos, masque_neg, omega=1.0, precision=1e-6, max_iter=100000):
    """
    Résolution de l'équation de Laplace par Gauss-Seidel avec relaxation (SOR).
    
    Paramètres
    ----------
    N         : int, taille de la grille
    masque_pos: ndarray bool, positions de la plaque +1 V
    masque_neg: ndarray bool, positions de la plaque -1 V
    omega     : float, paramètre de relaxation (1 = GS classique, >1 = sur-relaxation)
    precision : float, critère de convergence (norme inf du résidu)
    max_iter  : int, nombre maximum d'itérations
    
    Retourne
    --------
    V        : ndarray (N, N), potentiel convergé
    n_iter   : int, nombre d'itérations effectuées
    residus  : list, historique de convergence
    """
    V = np.zeros((N, N))
    V[masque_pos] = +1.0
    V[masque_neg] = -1.0
    # Bords à 0 (déjà initialisés)
    
    residus = []
    
    for iteration in range(max_iter):
        delta_max = 0.0
        
        # Mise à jour point par point (intérieur seulement)
        for j in range(1, N-1):
            for i in range(1, N-1):
                if masque_pos[i, j] or masque_neg[i, j]:
                    continue  # Conditions aux limites fixes
                
                # TODO: Implémenter la mise à jour Gauss-Seidel avec paramètre omega
                pass
        
        residus.append(delta_max)
        if delta_max < precision:
            return V, iteration + 1, residus
    
    print(f"Avertissement : convergence non atteinte après {max_iter} itérations.")
    return V, max_iter, residus

In [ ]:
# Calcul avec omega = 1 (Gauss-Seidel classique)
t0 = time.time()
V_gs, n_iter_gs, residus_gs = gauss_seidel(N, masque_pos, masque_neg, omega=1.0)
t_gs = time.time() - t0

print(f"Gauss-Seidel (ω=1.0) : {n_iter_gs} itérations en {t_gs:.2f} s")

# Visualisation : carte thermique
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

im = axes[0].imshow(V_gs.T, origin='lower', extent=[0, Lx, 0, Ly],
                    cmap='RdBu_r', vmin=-1, vmax=1)
plt.colorbar(im, ax=axes[0], label='Potentiel $V$ (V)')
axes[0].set_xlabel('x (cm)')
axes[0].set_ylabel('y (cm)')
axes[0].set_title('Potentiel électrostatique (Gauss-Seidel)')

# Lignes de champ (contours)
X, Y = np.meshgrid(x, y)
axes[1].contourf(X, Y, V_gs.T, levels=30, cmap='RdBu_r')
axes[1].contour(X, Y, V_gs.T, levels=15, colors='k', linewidths=0.5, alpha=0.5)
axes[1].set_xlabel('x (cm)')
axes[1].set_ylabel('y (cm)')
axes[1].set_title('Lignes équipotentielles')

plt.tight_layout()
plt.show()

# Convergence
fig, ax = plt.subplots(figsize=(9, 4))
ax.semilogy(residus_gs, color='#2196F3')
ax.axhline(1e-6, color='red', ls='--', label='Critère $10^{-6}$')
ax.set_xlabel('Itération')
ax.set_ylabel('Résidu maximal')
ax.set_title('Convergence de Gauss-Seidel')
ax.legend()
plt.tight_layout()
plt.show()

<div style="height: 30px;"></div>
<hr style="border: 0; height: 2px; background-color: #FF5722; border-radius: 2px; opacity: 0.6;">

## Question b — Effet du paramètre de relaxation $\omega$

<div style="background-color: #252526; color: white; border-left: 5px solid #FF5722; padding: 15px; border-radius: 5px;">

**Énoncé :** *Analysez l'effet du paramètre $\omega$ sur le temps de traitement de la méthode de Gauss-Seidel et déterminez la valeur optimale $\omega^*$ dans ce contexte.*

</div>

In [ ]:
# Balayage de omega
omegas = np.linspace(1.0, 1.99, 25)
temps_gs_omega = []
iters_gs_omega = []

for omega in omegas:
    t0 = time.time()
    _, n_iter, _ = gauss_seidel(N, masque_pos, masque_neg, omega=omega)
    t_elapsed = time.time() - t0
    temps_gs_omega.append(t_elapsed)
    iters_gs_omega.append(n_iter)
    print(f"ω = {omega:.3f} → {n_iter:6d} itérations, {t_elapsed:.2f} s")

omega_opt_idx = np.argmin(iters_gs_omega)
omega_opt = omegas[omega_opt_idx]
print(f"\n✓ omega optimal : {omega_opt:.3f} ({iters_gs_omega[omega_opt_idx]} itérations)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(omegas, iters_gs_omega, 'o-', color='#FF9800')
axes[0].axvline(omega_opt, color='red', ls='--', label=f'$\\omega^*$ = {omega_opt:.3f}')
axes[0].set_xlabel(r'Paramètre de relaxation $\omega$')
axes[0].set_ylabel("Nombre d'itérations")
axes[0].set_title("Convergence en fonction de $\omega$")
axes[0].legend()

axes[1].plot(omegas, temps_gs_omega, 's-', color='#FF5722')
axes[1].axvline(omega_opt, color='red', ls='--', label=f'$\\omega^*$ = {omega_opt:.3f}')
axes[1].set_xlabel(r'Paramètre de relaxation $\omega$')
axes[1].set_ylabel('Temps de calcul (s)')
axes[1].set_title('Temps de calcul en fonction de $\omega$')
axes[1].legend()

plt.tight_layout()
plt.show()

### Discussion

<div style="background-color: #1a2a3a; color: #e0e0e0; border-left: 5px solid #FF9800; padding: 15px; border-radius: 5px;">

La valeur optimale observée est $\omega^* \approx$ *[à compléter]*. On constate que ...

*Votre analyse ici...*

</div>

<div style="height: 30px;"></div>
<hr style="border: 0; height: 2px; background-color: #FF5722; border-radius: 2px; opacity: 0.6;">

## Question c — Méthode de Jacobi (avec et sans *slicing*) & comparaison

<div style="background-color: #252526; color: white; border-left: 5px solid #FF5722; padding: 15px; border-radius: 5px;">

**Énoncé :** *Refaites les calculs avec la méthode de Jacobi (avec et sans slicing). Comparez les temps de calcul en fonction de la taille de la grille pour les trois méthodes (Jacobi, Jacobi-slicing, Gauss-Seidel).*

</div>

In [ ]:
def jacobi_boucle(N, masque_pos, masque_neg, precision=1e-6, max_iter=100000):
    """
    Méthode de Jacobi — version boucle Python (sans slicing).
    """
    V = np.zeros((N, N))
    V[masque_pos] = +1.0
    V[masque_neg] = -1.0
    V_new = V.copy()
    
    for iteration in range(max_iter):
        # TODO: Mise à jour Jacobi (utilise V pour calculer V_new)
        pass
    
    return V, max_iter, []

In [ ]:
def jacobi_slicing(N, masque_pos, masque_neg, precision=1e-6, max_iter=100000):
    """
    Méthode de Jacobi — version vectorisée (avec slicing NumPy).
    
    Exploitation du découpage tableau pour éviter les boucles Python explicites.
    """
    V = np.zeros((N, N))
    V[masque_pos] = +1.0
    V[masque_neg] = -1.0
    
    residus = []
    
    for iteration in range(max_iter):
        # TODO: Mise à jour vectorisée Jacobi
        # V_new[1:-1, 1:-1] = (V[:-2, 1:-1] + V[2:, 1:-1] + ...) / 4
        pass
    
    return V, max_iter, residus

In [ ]:
# Comparaison en fonction de la taille de la grille
tailles = [20, 30, 40, 50, 60, 75, 100]
temps_jacobi_boucle = []
temps_jacobi_slice  = []
temps_gauss_seidel  = []

for n in tailles:
    mp, mn = construire_masque_plaques(n, Lx, Ly)
    
    t0 = time.time()
    jacobi_boucle(n, mp, mn)
    temps_jacobi_boucle.append(time.time() - t0)
    
    t0 = time.time()
    jacobi_slicing(n, mp, mn)
    temps_jacobi_slice.append(time.time() - t0)
    
    t0 = time.time()
    gauss_seidel(n, mp, mn, omega=omega_opt)
    temps_gauss_seidel.append(time.time() - t0)
    
    print(f"N={n:3d} | Jacobi-boucle: {temps_jacobi_boucle[-1]:.3f}s | "
          f"Jacobi-slice: {temps_jacobi_slice[-1]:.3f}s | "
          f"GS(ω*): {temps_gauss_seidel[-1]:.3f}s")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(tailles, temps_jacobi_boucle, 'o-', color='#e74c3c', label='Jacobi (boucle)')
ax.plot(tailles, temps_jacobi_slice,  's-', color='#f39c12', label='Jacobi (slicing)')
ax.plot(tailles, temps_gauss_seidel,  '^-', color='#2ecc71', label=f'Gauss-Seidel ($\\omega^*={omega_opt:.2f}$)')

ax.set_xlabel('Taille de la grille $N$', fontsize=12)
ax.set_ylabel('Temps de calcul (s)', fontsize=12)
ax.set_title('Comparaison des méthodes itératives — temps vs taille de grille')
ax.legend()
plt.tight_layout()
plt.show()

### Discussion et conclusions

<div style="background-color: #1a2a3a; color: #e0e0e0; border-left: 5px solid #FF9800; padding: 15px; border-radius: 5px;">

| Méthode | Avantages | Inconvénients |
|:--------|:----------|:--------------|
| Jacobi (boucle) | Simple à implémenter | Lent (boucles Python) |
| Jacobi (slicing) | Vectorisé, rapide | *À discuter* |
| Gauss-Seidel + $\omega^*$ | Convergence accélérée | *À discuter* |

*Votre analyse comparative ici...*

</div>

<div style="height: 50px;"></div>
<hr style="border: 0; height: 3px; background-color: #9C27B0; border-radius: 2px; opacity: 0.7;">
<br />

# Conclusions générales

<div style="background-color: #252526; color: white; border-left: 5px solid #9C27B0; padding: 15px; border-radius: 5px;">

### TP4.1 — Dynamique SIS

*Synthèse de vos résultats ici...*

- Les intégrateurs ont bien reproduit les ordres de convergence théoriques ($d = 1, 2, 4$).
- La méthode du ratio doré s'est révélée efficace pour cibler une précision donnée.

### TP4.2 — Condensateur électronique

*Synthèse de vos résultats ici...*

- La valeur optimale du paramètre de relaxation $\omega^*$ a permis de réduire significativement le temps de calcul.
- La méthode de Jacobi vectorisée (slicing) est ............ fois plus rapide que la version boucle.

</div>

<div style="height: 30px;"></div>
<hr style="border: 0; height: 2px; background-color: #607D8B; border-radius: 2px; opacity: 0.6;">

# Références

### Méthodes numériques

1. **Press, W.H. et al.** (2007). *Numerical Recipes: The Art of Scientific Computing* (3e éd.). Cambridge University Press.  
   *Référence générale pour l'intégration d'EDO et les méthodes de relaxation.*

2. **Burden, R.L. & Faires, J.D.** (2010). *Numerical Analysis* (9e éd.). Brooks/Cole.  
   *Analyse d'erreur globale et locale des méthodes de Runge-Kutta.*

### Modèles épidémiologiques

3. **Kermack, W.O. & McKendrick, A.G.** (1927). *A Contribution to the Mathematical Theory of Epidemics*. Proc. R. Soc. Lond. A 115, 700–721.  
   *Formulation originale des modèles compartimentaux SI/SIS/SIR.*

4. **Hethcote, H.W.** (2000). *The Mathematics of Infectious Diseases*. SIAM Review 42(4), 599–653.  
   *Revue complète incluant l'analyse des points fixes et du nombre de reproduction de base $R_0$.*

### Électrostatique numérique

5. **Griffiths, D.J.** (2017). *Introduction to Electrodynamics* (4e éd.). Cambridge University Press.  
   *Équation de Laplace, conditions aux limites et méthodes de relaxation.*

6. **Young, D.M.** (1954). *Iterative Methods for Solving Partial Difference Equations of Elliptic Type*. Trans. Amer. Math. Soc. 76, 92–111.  
   *Théorie de la sur-relaxation successive (SOR) et valeur optimale de $\omega$.*

### Cours et documentation

7. **Notes de cours PHY-3500** — Physique numérique (H26)  
   ↳ Professeurs : Antoine Allard, Thomas Labbé — Université Laval

8. **Énoncé du TP4** — Intégration numérique & Électrostatique  
   ↳ Version 2026-03-19


<div style="
    margin-top: 30px;
    padding: 15px;
    background: linear-gradient(135deg, rgba(15, 12, 41, 0.5), rgba(48, 43, 99, 0.3));
    border-radius: 8px;
    border: 1px solid rgba(168, 208, 230, 0.1);
    text-align: center;
">
    <p style="color: rgba(168, 208, 230, 0.5); font-size: 12px; margin: 0; letter-spacing: 1px;">
        PHY-3500 — Physique numérique (H26) — TP 4 : Intégration numérique & Électrostatique
    </p>
    <p style="color: rgba(168, 208, 230, 0.35); font-size: 11px; margin: 5px 0 0 0;">
        Alex Baker • Justine Jean • Nerimantas Caillat
    </p>
</div>